# Streaming

Calling agent.stream(...) yields chunks as the run unfolds, selected by stream_mode. Pick the granularity you need, or pass a list to receive several at once.

- "updates" — state changes after each agent step.
- "messages" — LLM tokens with metadata as they generate.
- "custom" — arbitrary signals you emit from inside a tool.

In [9]:
from pydantic import SecretStr
import os

from langchain.agents import create_agent
from langchain.tools import tool
from langchain.chat_models import init_chat_model

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"It's always sunny in {city}!"

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

agent = create_agent(
    model=model,
    tools=[get_weather]
)

for chunk in agent.stream(
    {"messages": [{"role":"user", "content": "What is the weather in SF?"}]},
    stream_mode="messages"):
    # print(chunk[0])
    if chunk[0]:
        print(chunk[0].content, end="", flush=True)

It's always sunny in San Francisco!San Francisco is currently sunny. The city is known for its mild, Mediterranean climate—cool, foggy mornings that often clear up to pleasant, dry afternoons. Temperatures typically stay in the 50‑60 °F (10‑20 °C) range in this season, with occasional breezes off the bay. If you’re heading out, a light jacket is usually a good idea.

## Emit your own progress from inside a tool with get_stream_writer():

In [14]:
from pydantic import SecretStr
import os

from langchain.agents import create_agent
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.config import get_stream_writer

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    writer = get_stream_writer()
    writer(f"looking up data for city: {city}")
    return f"It's always sunny in {city}!"

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

agent = create_agent(
    model=model,
    tools=[get_weather]
)

for chunk in agent.stream(
    {"messages": [{"role":"user", "content": "What is the weather in SF?"}]},
    stream_mode="custom"):
    print(chunk)
    # if chunk[0]:
    #     print(chunk[0].content, end="", flush=True)

looking up data for city: San Francisco


### Combine modes by passing a list, e.g. stream_mode=["updates", "custom"]; use agent.astream(...) for async I/O.